# Phase 3d - Band anchors for the zero-concentration boundary

The point anchor failed. 261 of 854 sequences (30.6%) still cross P = 0.5 at the
lowest tested concentration, and the anchored and unanchored response curves lie on
top of each other.

**Why it failed.** The anchors sat at exactly `Conc = 0`. A tree can place a single
split between 0 and the next representable value, satisfy every anchor, and leave all
real predictions untouched - monotonicity permits P(0) = 0 and P(0.0005) = 0.6. A
constraint imposed at a point is satisfied by a step.

**What the data says.** Only 11 rows in the whole dataset sit at exactly 0 uM, and
only 3 of those are positive, all from range expansion. So the model is not being
taught that zero protein demixes. The real cause is that the 1024 embedding
dimensions set the probability level and concentration only modulates it: for a
protein that demixes in most of its records, P stays high at any concentration.
Monotonicity constrains the slope, not the level.

**The fix.** Anchors spread over a low-concentration *band* rather than a point, with
enough weight to move the level. This notebook scans the weight.

**Setup:** T4 GPU, ~20 min. Upload `llpsense_final_noseq.csv` and `llpsense_t5.npz`.

## 1. Setup

In [ ]:
import numpy as np, pandas as pd, subprocess, time, json, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from scipy import stats
from xgboost import XGBClassifier
GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0

from google.colab import files
up = files.upload()
df  = pd.read_csv("llpsense_final_noseq.csv")
z   = np.load("llpsense_t5.npz", allow_pickle=True)
emb = pd.DataFrame(z["X"], index=z["SeqID"])
COND = ['Conc','pH','temp','MgCl2','NaCl','KCl','PEG300-1k','PEG3k-6k','PEG8k-20k',
        'Ficoll','Dextran -40','Dextran 70-','Glycerol']
X = np.hstack([emb.loc[df.SeqID].to_numpy(np.float32), df[COND].to_numpy(np.float32)])
y, w = df.Label.to_numpy(), df.w_record.to_numpy()
N_EMB = emb.shape[1]; IDX = {c: N_EMB+i for i,c in enumerate(COND)}
UP = ['Conc','PEG300-1k','PEG3k-6k','PEG8k-20k','Ficoll','Dextran -40','Dextran 70-']
MONO = tuple([0]*N_EMB + [1 if c in UP else 0 for c in COND])
HP = dict(n_estimators=877, max_depth=14, learning_rate=0.023722174554030036,
          min_child_weight=2, colsample_bytree=0.5655509178160505,
          subsample=0.5482175662874387, gamma=0.0012682360748828933,
          reg_lambda=0.3024802937837069, reg_alpha=0.04238171400820899, random_state=42)
RUNTIME = dict(tree_method="hist", eval_metric="logloss",
               **({"device":"cuda"} if GPU else {"n_jobs":-1}))
print("ready", X.shape)

## 2. Choosing the band

The band must not contradict observed positives. Below 0.1 uM the dataset holds 15
positive records, all arising from ranges whose lower endpoint approaches zero rather
than from measurements at 0.1 uM. A band of 0 to 0.1 uM is therefore defensible; the
cell below reports exactly what it conflicts with, so the choice is explicit.

In [ ]:
BAND_uM = 0.1
c_uM = df.Conc.to_numpy() * 1000.0
in_band = c_uM < BAND_uM
print(f"band 0 - {BAND_uM} uM")
print(f"  rows in band: {in_band.sum():,}  ({100*in_band.mean():.2f}% of data)")
print(f"  positive rows in band: {int(df.Label[in_band].sum())}"
      f"  from {df.RecordID[in_band & (df.Label==1)].nunique()} records")
print(f"  rows at exactly 0 uM: {(c_uM==0).sum()}  of which positive: {int(df.Label[c_uM==0].sum())}")

## 3. Weight scan

Anchors are real condition rows with concentration resampled uniformly inside the
band and the label set negative. The question is how much weight is needed before the
level actually moves.

In [ ]:
def band_anchored(Xb, yb, wb, frac, seed=0, band=BAND_uM):
    rng = np.random.default_rng(seed)
    pick = rng.choice(len(Xb), size=int(0.3*len(Xb)), replace=False)
    Xa = Xb[pick].copy()
    Xa[:, IDX['Conc']] = rng.uniform(0.0, band/1000.0, size=len(pick))   # normalised
    ya = np.zeros(len(Xa), dtype=int)
    wa = np.full(len(Xa), frac*wb.sum()/len(Xa))
    return np.vstack([Xb, Xa]), np.concatenate([yb, ya]), np.concatenate([wb, wa])

rep   = df.groupby("SeqID").head(1).index.to_numpy()
Xr    = X[rep].copy()
names = df.loc[rep, "ProteinName"].to_numpy()
cgrid_uM = np.logspace(np.log10(0.5), np.log10(563.0), 60)
cgrid    = cgrid_uM / 1000.0

def profile(model):
    P = np.zeros((len(Xr), len(cgrid)))
    for k, v in enumerate(cgrid):
        Xt = Xr.copy(); Xt[:, IDX['Conc']] = v
        P[:, k] = model.predict_proba(Xt)[:, 1]
    cs = np.full(len(Xr), np.nan)
    for i in range(len(Xr)):
        idx = np.where(P[i] > 0.5)[0]
        if len(idx): cs[i] = cgrid_uM[idx[0]]
    return P, cs

runs = {}
for frac in [0.0, 0.10, 0.25, 0.50, 1.00]:
    if frac == 0.0:
        m = XGBClassifier(**HP, monotone_constraints=MONO, **RUNTIME).fit(X, y, sample_weight=w)
        tag = "no anchor"
    else:
        Xa, ya, wa = band_anchored(X, y, w, frac)
        m = XGBClassifier(**HP, monotone_constraints=MONO, **RUNTIME).fit(Xa, ya, sample_weight=wa)
        tag = f"band anchor w={frac:.2f}"
    P, cs = profile(m)
    runs[tag] = (m, P, cs)
    at_floor = int(np.nansum(cs <= cgrid_uM[0]*1.001))
    print(f"{tag:22s}  mean P at 0.5 uM {P[:,0].mean():.3f}   "
          f"at floor {at_floor:3d}/{len(cs)} ({100*at_floor/len(cs):4.1f}%)   "
          f"median c_sat {np.nanmedian(cs):7.2f} uM", flush=True)

## 4. Did the level move?

In [ ]:
plt.figure(figsize=(7,5))
for tag, (m, P, cs) in runs.items():
    plt.plot(cgrid_uM, P.mean(0), lw=2, label=tag)
plt.axhline(.5, color="k", ls=":", lw=1)
plt.xscale("log"); plt.xlabel("protein concentration (uM)"); plt.ylabel("mean P(LLPS)")
plt.title("Concentration response vs anchor weight"); plt.legend(fontsize=9)
plt.tight_layout(); plt.savefig("band_anchor_scan.png", dpi=150); plt.show()

summary = pd.DataFrame({
    tag: {"mean P @0.5uM": round(float(P[:,0].mean()), 3),
          "at floor (%)": round(100*float(np.nansum(cs <= cgrid_uM[0]*1.001))/len(cs), 1),
          "median c_sat (uM)": round(float(np.nanmedian(cs)), 2),
          "c_sat defined": int(np.isfinite(cs).sum())}
    for tag, (m, P, cs) in runs.items()}).T
print(summary.to_string())
summary.to_csv("band_anchor_summary.csv")

## 5. What does the working weight cost?

Paired design, identical splits, anchors added to the training fold only. Set
`BEST` to the smallest weight in section 4 that actually moved the level.

In [ ]:
BEST = 0.50          # <-- set from section 4

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

for rname, key in [("A  new condition","RecordID"), ("B  new protein","SeqID")]:
    g, du, dc = df[key].to_numpy(), [], []
    for s in [1, 2]:
        for i, j in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(X, y, groups=g):
            if len(np.unique(y[j])) < 2: continue
            m0 = XGBClassifier(**HP, monotone_constraints=MONO, **RUNTIME).fit(X[i], y[i], sample_weight=w[i])
            Xi, yi, wi = band_anchored(X[i], y[i], w[i], BEST)
            m1 = XGBClassifier(**HP, monotone_constraints=MONO, **RUNTIME).fit(Xi, yi, sample_weight=wi)
            du.append(roc_auc_score(y[j], m0.predict_proba(X[j])[:,1], sample_weight=w[j]))
            dc.append(roc_auc_score(y[j], m1.predict_proba(X[j])[:,1], sample_weight=w[j]))
    du, dc = np.array(du), np.array(dc); d = dc - du
    t, p = stats.ttest_rel(dc, du)
    print(f"{rname:20s} monotone {du.mean():.4f} -> +band anchor {dc.mean():.4f}  "
          f"diff {d.mean():+.4f}  t {t:+.2f}  p {p:.3f}", flush=True)

In [ ]:
m, P, cs = runs[f"band anchor w={BEST:.2f}"]
pd.DataFrame({"SeqID": df.loc[rep,"SeqID"].to_numpy(), "ProteinName": names,
              "c_sat_uM": cs}).to_csv("csat_v3.csv", index=False)
from google.colab import files
files.download("csat_v3.csv"); files.download("band_anchor_summary.csv")
files.download("band_anchor_scan.png")

### Reading section 4

If the floor percentage falls steadily with weight, the band formulation works and the
scan tells you the price. If it stays near 30% even at weight 1.0, the conclusion is
stronger and simpler: **sequence embeddings dominate the probability level so completely
that condition-based constraints cannot correct it** - which would say something real
about this architecture, not just about anchors.